# Exploratory Data Analysis In-Class Activity: Exploring Civil War Onset Indicators



**Introduction:** In this tutorial, we will use Python and the pandas library to explore a real dataset on the causes of civil wars. The dataset (known as the [Collier-Hoeffler Civil War Dataset](https://documents1.worldbank.org/curated/en/359271468739530199/pdf/multi-page.pdf), one of the most influential datasets in conflict studies) contains information about various country-level characteristics and whether a civil conflict began in each five-year period. Each row represents a country during a specific 5-year interval (for example, a row for **Afghanistan, 1960** covers the period 1960–1965). Our goal is to practice fundamental EDA operations in pandas (loading data, inspecting data frames, calculating summary statistics, filtering, grouping, creating new variables) while reasoning about which factors might be associated with the outbreak of civil war. We will examine economic indicators (like export dependence, education, and economic growth) and social/demographic indicators (like population, ethnic fractionalization, and dominance) in relation to civil war onsets. We will also reflect on our findings and consider potential biases or pitfalls in interpreting this data.

**Dataset features:** The dataset includes the following columns (variables) for each country-period combination:

- **country** – Country name (each country may appear in multiple 5-year periods).
- **year** – Starting year of the five-year interval (e.g., 1960, 1965, 1970, ...).
- **start** – Indicator of civil war onset during that period (**1** if a new civil war started, **0** if no new war; **NA** denotes that a war was ongoing through the period, so no new war started in that interval).
- **exports** – Dependence on primary commodity exports (a measure of how much the country’s economy relies on commodities; higher values mean a greater share of income from commodity exports).
- **schooling** – Male secondary school enrollment rate (in percent of the relevant age group).
- **growth** – Annual GDP growth rate (percentage growth of the economy per year).
- **concentration** – Population concentration index (ranges from 0 to 1; 1 means the entire population lives in one city/area, 0 means population is evenly spread out).
- **peace** – Number of months of peace since the country’s last civil war (or since the end of WWII if the country has not had a civil war in the interim).
- **lnpop** – Natural logarithm of the country’s total population.
- **fractionalization** – An index of social fractionalization (reflecting how ethnically and/or religiously divided the society is). Higher values indicate a more fractionalized (diverse) society. *(In the data this index is provided on a scaled numeric scale; we will treat larger numbers as indicating more diversity.)*
- **dominance** – An index of ethnic dominance (a binary indicator of whether one ethnic group is dominant in the country’s population). Typically defined such that 1 indicates a single ethnic group makes up a large share of the population (for example, dominance = 1 if one group comprises a majority but not nearly the entire population, and 0 otherwise).

Throughout this notebook, we will walk through steps to **load and inspect the data**, **explore the distribution of civil war onsets across time and countries**, **analyze relationships** between war onset and various economic and social factors, and **practice drawing interpretations** from data. We will explain each command and concept. Remember to run each code cell in order, and feel free to add your own code or Markdown cells to explore further or note your thoughts.

*Let’s get started!*



## Learning Objectives



By the end of this tutorial, you will be able to:

1. **Load data** from a CSV file into a pandas DataFrame.
2. **Inspect and summarize** a DataFrame’s structure (rows, columns, data types, and basic statistics).
3. **Clean the data** by handling unnecessary columns and identifying missing values.
4. Perform **exploratory data analysis (EDA)** using filtering, grouping, and aggregation to find patterns in the data.
5. **Engineer new features** to derive more meaningful information (e.g., transforming or combining columns).
6. Create basic **data visualizations** (line plots, bar charts, histograms, scatter plots) to reveal trends and distributions.
7. Understand the basics of **statistical inference** (e.g., comparing groups and considering significance and causation).
8. **Communicate results** effectively, interpreting data patterns and considering context, biases, and limitations.
9. Tackle a **capstone project** by applying these skills to a new question or analysis on the dataset.
10. Recognize common **troubleshooting** issues and approaches when working on a data science project with pandas.

## 1. Data Loading



First, we need to load the data into a pandas DataFrame. The dataset is stored in a CSV file named `ch.csv`. If you are running this notebook locally, make sure `ch.csv` is in your working directory (or adjust the path). If you are working in an online environment, you might need to upload the file or use a provided path/URL. In pandas, the **`pd.read_csv()`** function is used to read a CSV file into a DataFrame.

**Learning objectives for this section:** Learn how to import pandas, load a CSV file into a DataFrame, and create an initial DataFrame object for analysis.

**Task:** Execute the code below to load the dataset into a DataFrame called `df`. Then we'll do some quick checks to confirm the data loaded correctly.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # Turn off distracting warnings

import pandas as pd

# Load the dataset
df = pd.read_csv('ch.csv')
# Peek at the first five rows
df.head()

After running `df.head()`, you should see output similar to the following (showing the first few rows):

```
   Unnamed: 0      country  year  start  exports  schooling  growth  peace  \
0           1  Afghanistan  1960    0.0    0.074        2.0     NaN  172.0   
1           2  Afghanistan  1965    0.0    0.074        4.0     NaN  232.0   
2           3  Afghanistan  1970    0.0    0.043       13.0     NaN  292.0   
3           4  Afghanistan  1975    1.0    0.085       13.0     NaN  352.0   
4           5  Afghanistan  1980    NaN    0.240       16.0     NaN    NaN   

   concentration     lnpop  fractionalization  dominance  
0          0.492  16.11969              132.0        1.0  
1          0.492  16.22381              132.0        1.0  
2          0.492  16.33779              132.0        1.0  
3          0.492  16.45728              132.0        1.0  
4          0.492  16.58497              132.0        1.0  
```

Let’s interpret what we see in these first few rows:

- There is an index column labeled `Unnamed: 0` that appears to be just row numbers from the original source. We do not need this as a separate column, and we will remove it soon to avoid confusion.
- Each row has values for **country** and **year**. In the sample above, all rows shown are for Afghanistan in consecutive periods (1960, 1965, 1970, 1975, 1980).
- The **start** column shows values 0.0, 1.0, or NaN. For example, in 1975 for Afghanistan, `start = 1.0`, indicating a civil war began in that period. In 1960, 1965, 1970 `start = 0.0` (no new war). In 1980, `start = NaN`, which (as explained in the dataset description) means a war was ongoing during that period (so the war that started in 1975 was still happening through 1980–1985, and thus no *new* war started in 1980).
- Other columns show various numeric values or `NaN` for missing data. For instance, Afghanistan’s **growth** (GDP growth) is `NaN` in these early periods (perhaps due to missing economic data for those years), and **peace** (months of peace) resets to `NaN` in 1980 when a war was ongoing (since you can’t count peaceful months during a war).

Now that the data is loaded, let's move on to inspecting its structure in more detail.




**Your turn (Q1):** *How many rows and columns does the DataFrame have? What does each row represent in this dataset?*

**Your turn (Q2):** *In the snippet above, what does a `NaN` value in the `start` column represent, and why might some periods have `start = NaN`?*

*(You can answer these questions in a Markdown cell after thinking about the data.)*

**Answer (Q1):**



**Answer (Q2):**



## 2. Data Inspection



Before diving into analysis, it's important to get a sense of the dataset's overall shape and structure. In this section, we will check the size of the dataset, the data types of each column, and see how much missing data there is.

**Learning objectives for this section:** Use pandas attributes and methods to inspect a DataFrame (such as `.shape`, `.columns`, `.info()`, and `.describe()`), and understand the concepts of data types and missing values.

Let's first check the dimensions of the DataFrame and the column names:


In [ ]:
# Inspect basic structure
print("Number of rows and columns:", df.shape)
print("Column names:", df.columns.tolist())

When you run this, `df.shape` will output the tuple *(1288, 12)* indicating the DataFrame has 1288 rows and 12 columns (in our case, 11 data columns plus the extra `Unnamed: 0` index column). The list of column names will also be printed.

Next, we'll use **`df.info()`** to get a concise summary of the DataFrame:

In [ ]:
# Drop the redundant index column if present, then inspect the DataFrame's structure
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df.info()

We dropped the `Unnamed: 0` column, as it was just a duplicate index from the file. Now `df.info()` provides useful information about the DataFrame:

- The number of entries (rows) and columns.
- The name and data type of each column.
- The count of non-null (non-missing) values in each column.

You should see something like this:

```

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1288 entries, 0 to 1287
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   country            1288 non-null   object
 1   year               1288 non-null   int64
 2   start              1167 non-null   float64
 3   exports            1163 non-null   float64
 4   schooling          1037 non-null   float64
 5   growth             918 non-null    float64
 6   peace              1167 non-null   float64
 7   concentration      1128 non-null   float64
 8   lnpop              1266 non-null   float64
 9   fractionalization  1160 non-null   float64
 10  dominance          1184 non-null   float64
dtypes: float64(9), int64(1), object(1)
memory usage: 110.8+ KB

```

Key takeaways from this output:

- **Dimensions**: 1288 rows, 11 columns (after dropping the redundant index column).
- **Data types**: Most columns are `float64` (numeric), except `country` (object, i.e. string) and `year` (int64).
- **Non-null counts**: You can see how many non-missing values are in each column. For example, `schooling` has 1037 non-null entries out of 1288, meaning there are 1288 - 1037 = 251 missing values for schooling. Likewise, `growth` has only 918 non-null, so about 370 values of `growth` are missing. Other columns like `start`, `exports`, `peace`, etc., also have some missing values (NaNs).

It's often useful to get some basic summary statistics of the numeric columns as well. We can use **`df.describe()`** for that:

This will output count, mean, standard deviation, and quartile statistics for each numeric column (for instance, you might see the mean `start` is around 0.07, which makes sense because relatively few periods have a war onset (1) compared to no new war (0)). The `describe()` output can help identify ranges and potential outliers in the data.

From the inspection above, we note that several columns have missing data (NaNs). For example, **growth** is missing a lot of values (perhaps GDP growth data wasn’t available for many country-years). We will need to keep these missing values in mind during analysis and possibly handle them.

At this point, we have:

- Loaded the data into `df`.
- Removed an unnecessary column.
- Viewed the first few rows to confirm the content.
- Checked the overall structure and noted where data is missing.



**Your turn (Q3):** *Which columns have the most missing values? How might missing data affect our analysis, and what are two possible strategies to handle missing data in a dataset like this?*

*(Think about this and write your thoughts in a Markdown cell. For instance, you might note that **growth** has a lot of missing entries, and strategies could include dropping those rows or imputing approximate values, each with pros/cons.)*

**Answer (Q3):**



## 3. Data Cleaning



Data cleaning involves fixing or mitigating issues in the dataset such as missing values, incorrect data types, or extraneous information. In our dataset, we have already addressed one cleaning task (removing the redundant index column). Now let's consider how to handle missing data and any other cleaning needed.

**Learning objectives for this section:** Learn ways to detect and handle missing data in pandas, and perform basic cleaning steps like dropping or filling missing values and converting data types if necessary.

First, let's quantify the missing data more clearly. We can use **`df.isnull().sum()`** (or `df.isna().sum()`, which is equivalent) to see the number of missing entries per column:

In [ ]:
df.isnull().sum()

This will output something like:

```

country               0
year                  0
start               121
exports             125
schooling           251
growth              370
peace               121
concentration       160
lnpop                22
fractionalization   128
dominance           104
dtype: int64

```

This confirms what we saw in `df.info()`: for example, **growth** has 370 missing values, and **schooling** has 251 missing, etc. The columns `country` and `year` have no missing values (as expected for how the data was collected).

How should we deal with these missing values? The answer depends on our analysis goals:

- Since this is an exploratory analysis, we can **leave missing values as is** for now, because pandas functions like `.mean()` or `.groupby()` will automatically skip NaNs in calculations. We will just be careful to note if missing data might bias a result (for instance, if war-torn periods are more likely to have missing economic data, that could affect comparisons).
- Alternatively, we could **drop rows** that have missing values if we needed a complete-case analysis (but dropping ~370 rows for growth might remove a lot of information, possibly biasing toward certain years or countries).
- Or we could **impute** (fill in) missing values with something like a column mean or median, but for this exercise, we won't do that because it could distort genuine differences (and would require careful justification).

For now, we will proceed without aggressive imputation. We'll be mindful of missing data when interpreting results. If a certain calculation involves a column with missing values, pandas will handle it by default (for example, `df['growth'].mean()` will compute the mean of available growth values, ignoring NaNs).

Another minor cleaning task could be checking data types. For instance, `dominance` is a float column with values 0.0 or 1.0, which could logically be considered a boolean or integer. However, this is not critical to change; the float representation is fine for analysis (just something to note).

Now that our dataset is loaded, inspected, and cleaned of obvious issues, we can move on to exploring the data!

*(No specific code to run in this section aside from optional missing data handling. We'll address any needed cleaning as we encounter issues in analysis.)*



**Your turn (Q4):** *Why is it important to be aware of missing data before doing analysis? What could happen if we ignore a column’s missing values when comparing groups or computing averages?*

**Answer (Q4):**





## 4. Exploratory Data Analysis (EDA)



Now we dive into exploring the dataset to uncover patterns and relationships. A key question we want to answer is: **How are civil war onsets distributed across time and across countries?** Understanding the occurrence of conflicts in our data will provide important context for further analysis.

We will start by examining the **`start`** variable (civil war onset indicator) to see how many instances of war onset vs peace we have, and how these war onsets are spread over different years and countries.

**Learning objectives for this section:** Practice using pandas methods for data exploration, such as `value_counts()`, boolean indexing (filtering), and `groupby` for aggregation. Gain insights into the distribution of the target variable (`start`) across time and entities.

### 4.1 Frequency of civil war onsets vs. peace periods

First, let's see the overall frequency of war onsets in the dataset compared to peaceful periods. Remember:

- `start = 1` means a **new civil war started** in that 5-year period.
- `start = 0` means **no new war** started (the country remained in peace during that period, not counting any ongoing war from before).
- `start = NaN` means the country was in an **ongoing war** throughout that period (so no new war *could* start because one was already happening).

We can use `value_counts()` to tabulate the occurrences of each value in `start`. We'll include `dropna=False` to count the `NaN` entries as well:

In [ ]:
df['start'].value_counts(dropna=False)

This will output something like:

```

0.0    1089
NaN     121
1.0      78
Name: start, dtype: int64

```

Interpreting this:

- **1089** entries have `start = 0.0`, meaning in 1089 country-periods there was **no new war** (peace persisted during those periods).
- **78** entries have `start = 1.0`, meaning there were 78 distinct civil war **onset events** recorded in the data.
- **121** entries are `start = NaN`, meaning 121 country-periods were times during which a war was ongoing (these are not counted as new onsets; they indicate continued conflict from a previous period).

These numbers tell us that new civil wars were relatively infrequent in the dataset (78 out of 1288 observations). Many more observations are peaceful periods, and a significant number are periods of ongoing war that started earlier. In other words, our data has far more "peace" years than new conflict onsets, which is important to keep in mind.



### 4.2 War onsets over time



Next, let's examine how those 78 war onsets are distributed over the years 1960–1995. We might suspect that certain historical periods had more conflicts starting than others.

To see this, we can group the data by the **year** and count how many war onsets happened in each five-year interval. One way to do this:

1. Filter the DataFrame to include only rows where `start == 1` (war start events).
2. Group by `year` and count the number of occurrences.

Let's do that:

In [ ]:
# Filter to war onset events
war_onsets = df[df['start'] == 1.0]

# Count war onsets per five-year period (year)
war_counts_by_year = war_onsets.groupby('year').size()
war_counts_by_year

The result (a pandas Series) will list each 5-year interval present in the data and the number of wars that began in that interval. For example, you might see output like:

```

year
1960    11
1965     7
1970    11
1975    10
1980    12
1985     6
1990    16
1995     5
dtype: int64

```

This indicates:

- In the period **1960–64**, there were 11 civil wars that started across all countries.
- 1965–69: 7 new wars.
- 1970–74: 11 new wars.
- 1975–79: 10 new wars.
- 1980–84: 12 new wars.
- 1985–89: 6 new wars.
- 1990–94: 16 new wars (this appears to be the peak).
- 1995–99: 5 new wars (note: the interval labeled 1995 in the data would cover 1995–1999, but our dataset stops at 1995 as the start year, so it likely only includes up to 1999).

We can see that the **early 1990s (the period starting 1990) had the highest number of new civil conflicts** in this dataset (16 outbreaks). Historically, this corresponds to the post-Cold War era and the collapse of the Soviet Union/Yugoslavia, when many new conflicts emerged. In contrast, the late 1980s saw relatively fewer new wars (only 6 starting in 1985–89). The 1960s and 1970s each had around 10–11 new wars per period, and the mid-1990s had fewer (possibly because our data truncates at 1999, or indeed there were fewer new wars as some conflicts wound down by then).



### 4.3 War onsets across countries



Now, let's examine war onsets **across countries**. We want to know how many conflicts each country experienced over this timeframe. Did most countries have zero or one civil war onset, or did some have multiple?

We can find this by counting how many times each country has `start = 1`. Essentially, count war onsets per country:

In [ ]:
# Count of war onsets per country
war_counts_by_country = war_onsets['country'].value_counts()
# Display the top 10 countries by number of war onsets
war_counts_by_country.head(10)

This will list the countries with the most civil war start events. For example, you might get something like:

```
country
Iraq             3
Angola           3
Zaire            3
Burundi          3
Iran             3
Rwanda           2
Algeria          2
Mozambique       2
Myanmar/Burma    2
Nicaragua        2
Name: country, dtype: int64

```

From this sample, we see a few countries had multiple separate civil war onsets during 1960–1995:

- Countries like **Iraq, Angola, Zaire (Democratic Republic of Congo), Burundi, Iran** each had **3** distinct civil wars begin in this period. This suggests cycles of conflict (a new war, then peace, then another war later).
- Several countries had 2 onsets (**Rwanda, Algeria, Mozambique, Myanmar/Burma, Nicaragua**, etc.).
- Most countries (not listed in the top 10) had 0 or 1 war onset. If you were to count beyond the top 10, you'd find that the majority of the countries in the dataset did not experience civil war onset more than once. Only a smaller set of countries went through repeated conflicts.

This highlights that civil wars tend to be **concentrated**: certain countries faced multiple outbreaks of conflict, whereas many others remained peaceful (at least in terms of no new civil war onset during 1960–1995, even if some had ongoing wars that started before 1960 in a couple of cases).

Now that we have a sense of **when** and **where** conflicts happened in our data, let's reflect on these patterns.

**Interpreting the patterns:**

- **Over time:** The frequency of new wars was not constant. There was a notable spike around the early 1990s. This could be due to geopolitical shifts (e.g., regime changes, end of the Cold War, breakup of multi-ethnic states, etc.). Earlier periods like the 1960s and 1970s also saw multiple new conflicts, often related to decolonization and Cold War proxy wars.
- **Across countries:** Relatively few countries suffered repeated civil wars in this timeframe. Many countries never had a civil war onset in these data, or had only one onset. However, a handful of countries experienced conflict multiple times, indicating unstable conditions or recurring tensions in those places (for example, Angola had a long civil war with ceasefires and restarts; Iran experienced multiple distinct conflicts if we consider revolutions and regional strife; etc.).
- The `NaN` entries for `start` (ongoing wars) correspond to long wars. For example, in our earlier peek, Afghanistan in 1980 had `start = NaN`, meaning the war that started in 1975 was still ongoing through the 1980–85 interval. Countries with `NaN` in multiple consecutive periods indicate **protracted conflicts** (a war spanning more than 5 years).



**Your turn (Q5):** Which 5-year interval saw the highest number of new civil wars? Which interval saw the fewest? What historical events might explain these observations?

**Your turn (Q6):** Name one or two countries that experienced multiple civil war onsets. What does this suggest about the stability of those countries during 1960–1995?

*(Provide your answers in a Markdown cell below, using evidence from the counts we calculated. You don’t need deep historical knowledge—focus on the patterns in the data. For example, you could note that 1990–94 had the most onsets and connect it to global political changes, or mention that Angola had 3 onsets indicating recurring conflict.)*

**Answer (Q5):**



**Answer (Q6):**



## 5. Feature Engineering



So far, our analysis has used the variables as given in the dataset. Feature engineering is the process of creating new variables (features) or transforming existing ones to help reveal patterns or make analysis more intuitive. We will create a couple of new features to aid our analysis of social and demographic factors next.

**Learning objectives for this section:** Practice creating new columns in a DataFrame using transformations of existing columns, and understand the rationale for feature engineering (e.g., making variables more interpretable or useful for analysis).



### 5.1 Creating new features for clarity



Looking at the dataset, one variable that could use a more intuitive scale is **population**. We have `lnpop` (the natural log of population). It might be easier to discuss population in millions of people rather than in log form. We can create a new column for **population in millions** by exponentiating `lnpop` (since if `lnpop = log(population)`, then `population = exp(lnpop)`).

Also, consider **GDP growth**: sometimes it’s useful to flag whether a country was in a recession (negative growth). We can engineer a feature for that as well, which might be interesting to see in relation to war onset.

Let's create two new features:

- `population_millions` – estimated population of the country in that period, measured in millions.
- `recession` – a boolean (True/False) indicating whether the GDP growth was negative (i.e., an economic recession).

In [ ]:
import numpy as np

# Create a new column for population (in millions) by exponentiating lnpop
df['population_millions'] = np.exp(df['lnpop']) / 1_000_000

# Create a new column indicating if the growth was negative
df['recession'] = df['growth'] < 0

# Check the new columns for the first few rows
df[['country', 'year', 'lnpop', 'population_millions', 'growth', 'recession']].head(10)


After this, our DataFrame `df` has two new columns: **`population_millions`** and **`recession`**. In the snippet of the first 10 rows, you'll see `population_millions` for Afghanistan 1960–1980, and whether each period had a recession (False/True). Note that if `lnpop` was missing for a row, `population_millions` will be `NaN` as well.

Why create these features?

- By converting log-population to actual population in millions, we can say things like "Country X had ~50 million people" instead of "lnpop was ~17". This is more tangible.
- The `recession` flag simplifies analysis of economic conditions: instead of comparing exact growth rates, we can easily count or group by whether growth was negative or not, to see if wars are more likely during recessions.

*(Creating these new features did not produce textual output, except for our head() check. They will be used in the next section.)*




**Your turn (Q7):** Why might we want to use `population_millions` instead of `lnpop` when communicating our findings? Can you think of another new feature that could be useful in this dataset (for example, a categorical version of a numeric variable)?

*(Answer in Markdown: for instance, you could mention that population in millions is easier to interpret for storytelling. Another feature idea could be grouping schooling into “low/medium/high education” categories, or converting peace months into peace years.)*

**Answer (Q7):**



## 6. Data Visualization (Basic Plotting)



Visualizing data can often reveal patterns more clearly than tables of numbers. We will create a few simple plots to complement our analysis. We'll use the basics of Altair for quick visualizations, but will spend more time learning advanced features of Altair in future weeks.

**Learning objectives for this section:** Learn how to produce basic plots using Altair, such as bar charts, line charts, histograms, and scatter plots, and interpret them in the context of the data.



### 6.1 Plotting war onsets over time



We earlier aggregated the number of war onsets per five-year period. We can visualize that time trend with a simple bar chart or line chart.

Let's use the `war_counts_by_year` Series we computed and make a bar chart of war onsets by period:

Before we can create our visualization with Altair, we need to convert our pandas Series to a DataFrame. Altair works with tabular data where each row represents an observation and each column represents a variable. Our Series has years as the index and counts as values, but Altair needs both of these as separate columns in a DataFrame. We can accomplish this conversion using the `reset_index()` method, which turns the index into a regular column, creating the two-column structure that Altair expects.

In [ ]:
# Convert Series to DataFrame
war_counts_by_year_df = war_counts_by_year.reset_index()
war_counts_by_year_df.columns = ["Year", "War Count"]

import altair as alt

# Bar chart of civil war onsets by 5-year period
alt.Chart(war_counts_by_year_df, title="Civil War Onsets Over Time").mark_bar(size=20).encode(
    alt.X("Year").title("Start Year of 5-year interval"),
    alt.Y("War Count").title("Number of new wars")
)

Running this will produce a bar chart where the x-axis is the start year of the interval (1960, 1965, ..., 1995) and the y-axis is the count of new wars that started in that interval. The chart should show a peak at 1990, a dip around 1985, and moderate values for other years, consistent with the numbers we saw.

If we prefer a line chart (since the intervals have an order), we could do:

In [ ]:
# Line chart of civil war onsets by 5-year period
alt.Chart(war_counts_by_year_df, title = "Civil War Onsets Over Time").mark_line(point=True).encode(
    alt.X("Year").title("Start Year of 5-year interval"),
    alt.Y("War Count").title("Number of new wars")
)

This will connect the points in chronological order. Either way, the visualization confirms the spike in the early 1990s.



### 6.2 Distribution of a numeric variable (histogram)



Next, let's visualize the distribution of an indicator. For example, **schooling** (male secondary education %). We can see how education levels are distributed across country-periods.

In [ ]:
# Histogram of schooling percentages
alt.Chart(df, title="Distribution of Schooling Levels").mark_bar().encode(
    alt.X("schooling", bin=alt.BinParams(maxbins=20)).title("Male secondary schooling (%)"),
    alt.Y("count()").title("Number of country-periods")
)

This histogram will show how many observations fall into various schooling rate bins. We might see a bimodal or skewed distribution – for instance, some country-periods with very low schooling (under 20%) and a decent number with quite high values (60-100%). If the distribution is multi-peaked, it could reflect different groups of countries (e.g., more developed vs less developed).

You can similarly plot histograms for other continuous variables like `growth` (GDP growth rates) to see their distribution (likely centered around a few percent growth, with a tail into negative).



### 6.3 Scatter plot to examine relationships



To explore relationships between two variables, scatter plots are useful. Let's see if there's any relationship between **education and economic growth** in this dataset (e.g., do periods with higher schooling tend to have higher growth?).

In [ ]:
# Scatter plot of schooling vs GDP growth
alt.Chart(df, title = "Schooling vs GDP Growth").mark_circle().encode(
    alt.X("schooling").title("Male secondary schooling (%)"),
    alt.Y("growth").title("Annual GDP growth (%)")
)

Each point in this plot represents a country-period. We might expect that higher schooling could correlate with higher growth, but the data may not show a strong pattern (especially because many other factors affect growth). If the points are very dispersed with no clear trend, that indicates little linear correlation between these two variables in our dataset. And indeed, we might see a cloud of points without a strong upward or downward slope.

Another interesting scatter could be between **population size and fractionalization** (are larger countries more diverse?) or **exports and growth** (to see if commodity dependence relates to growth rates). You can try:


In [ ]:
# Scatter plot of exports vs GDP growth
alt.Chart(df, title="Exports vs GDP Growth").mark_circle().encode(
    alt.X("exports").title("Primary commodity exports (fraction of GDP)"),
    alt.Y("growth").title("Annual GDP growth (%)")
)

These also help practice making plots, even if interpreting them requires caution. (For example, the exports vs growth might show if high commodity dependence countries have volatile or lower growth — a common hypothesis, but our data might not conclusively show it.)



### 6.4 Comparing groups with visuals



We might also want to compare distributions for war vs peace periods visually. One way is using a **box plot** to show the distribution of a variable for two groups. Pandas can do this via `DataFrame.boxplot` grouping by a category. For instance, to compare schooling in war-onset vs peaceful periods:
Note: If your data contains missing values in the grouping variable, you may want to filter them out first using `transform_filter()` to avoid displaying an unwanted "null" category in your visualization.

In [ ]:
# Box plot comparing schooling levels between war onset and peaceful periods
alt.Chart(df, title = "Schooling vs. War Onset").transform_filter(
    "datum.start != null"
).mark_boxplot().encode(
    alt.X("start:N").title("start (0 = no war, 1 = war onset)"),
    alt.Y("schooling:Q").title("Male secondary schooling (%)")
)

This will produce side-by-side boxplots: one for `start=0` (peace periods) and one for `start=1` (war onset periods). From our earlier analysis, we expect the median schooling in war onset periods to be lower than in peace periods. The boxplot can illustrate this difference (the median line in the war box might be lower, and the spread could also differ).

Similarly, we could boxplot GDP growth by start=0/1 to visualize that war periods tend to have lower (even negative) growth. And we could do fractionalization by start to see if war periods have higher diversity distribution.

Visualizing the comparisons can reinforce the numeric summaries we will do in the next section, and it's a good check for outliers or distribution shape differences (e.g., maybe a few war cases had very high schooling which could appear as outliers on the plot).



**Your turn (Q8):** Look at the charts you've generated (war onsets over time, schooling distribution, etc.). What stands out to you? For example, describe the trend shown in the war onsets over time plot, or any skew you notice in the schooling histogram. *What might these visual patterns imply about the data?*

*(Write a short interpretation in Markdown. For instance: "The bar chart of war onsets over time clearly peaks in 1990, suggesting a global wave of conflicts in the early 90s. The schooling histogram shows a lot of country-periods with very low schooling rates, implying many countries had low secondary education in earlier decades, whereas a smaller set of observations have near 100% schooling (likely wealthier countries).")*

**Answer (Q8):**




## 7. Statistical Inference



Up to now, our analysis has been exploratory and descriptive. We observed differences (e.g., periods with war onset had lower average schooling and GDP growth than peaceful periods). **Statistical inference** allows us to assess how likely these patterns are to be due to chance and helps us be cautious about drawing conclusions.

In this section, we'll briefly illustrate how we might test whether the differences we observed are statistically significant, and discuss the relationship between correlation and causation.

**Learning objectives for this section:** Understand how to perform a simple statistical test (like a t-test) to compare groups, interpret p-values in context, and reinforce the idea that correlation does not imply causation.



### 7.1 Testing group differences



**Example: Education and civil war onset.** We found that the average secondary schooling was lower in war-onset periods (~30%) compared to peaceful periods (~44%). Is this difference likely due to random variation, or is it statistically significant?

We can perform an independent two-sample t-test on the schooling percentages of the two groups (war vs peace). We'll use SciPy for this:


In [ ]:
from scipy import stats

# Separate schooling values for war onset vs peace (dropping NaNs)
schooling_war = df[df['start'] == 1.0]['schooling'].dropna()
schooling_peace = df[df['start'] == 0.0]['schooling'].dropna()

# T-test for difference in means
t_stat, p_value = stats.ttest_ind(schooling_war, schooling_peace, equal_var=False)
print("T-statistic:", round(t_stat, 3), "P-value:", p_value)

This will output a t-statistic and a p-value. For instance, you might see something like:

```

T-statistic: -4.172  P-value: 7.67e-05

```

The **p-value ~ 7.7e-05** (0.000077) is very low – far below a typical significance level like 0.05. This suggests that the difference in schooling between war and peace periods is statistically significant (i.e., it's very unlikely to see a gap this large if in truth war vs peace made no difference).

We could do similar tests for other variables:

- GDP **growth**: likely also significantly lower in war periods (our test would yield a low p-value as well).
- **Fractionalization**: war periods had higher average fractionalization; a t-test gave p ~ 0.02 in our calculations, indicating a significant difference (though less extreme than schooling).
- **Dominance**: we saw virtually no difference; indeed, a test yields a p-value ~0.8 (no significant difference in dominance between war vs peace cases).

For categorical comparisons like **dominance (0/1)** vs **war (0/1)**, we could also do a chi-square test on a contingency table. For example:

In [ ]:
contingency = pd.crosstab(df['dominance'].fillna(-1), df['start'].fillna(-1))
contingency  # Just to see the table

The table might look like (using -1 to denote the NaN category if any):

```

start       -1   0.0   1.0
dominance
0.0          0   549    40
1.0          0   478    33

```

This means when dominance=0 (no ethnic majority), 40 out of 589 such cases had a war (about 6.8%). When dominance=1, 33 out of 511 cases had a war (~6.5%). These percentages are virtually the same, which matches our earlier observation. A chi-square test would confirm no significant dependence between dominance and war onset.

The broader point is: **Yes, the differences we saw in schooling, growth, and fractionalization are supported by statistical tests as unlikely to be due to random chance.** However, that does not imply a causal relationship; it only confirms that the associations are statistically robust.



### 7.2 Correlation vs. causation



Throughout our analysis, we've been careful to say things like "associated with" or "correlates with". Why? Because even if, for example, low schooling is correlated with civil war onset, it doesn't mean low schooling *causes* civil war. There are many possible explanations:

- **Confounding factors:** Perhaps low schooling and civil war both result from a third factor like poverty or weak government institutions.
- **Reverse causation:** In some cases, the causal direction might be opposite or a feedback loop. For instance, a war could disrupt schooling (so the war causes low schooling in that period, not the other way around). Similarly, a war onset could cause an economic recession (so the war leads to negative growth in that year, rather than negative growth leading to war). Our data is structured per period, making timing subtle. If a war starts in 1975, the GDP growth for 1975 might already be affected by that conflict starting.
- **Sampling bias:** Missing data or the selection of countries could bias results. (Maybe the data missingness is not random – e.g., countries with war might have missing economic data, which could skew simple comparisons.)

It's crucial to remember that **correlation does not imply causation**. Statistical inference (like t-tests, regression, etc.) can tell us if differences or correlations are likely real (not random), but determining causation requires theory, careful study design, or more advanced methods (like looking at changes over time, using instrumental variables, etc., beyond our scope here).



### 7.3 Optional: Toward predictive modeling



If we were to continue the data science pipeline, a next step could be building a simple **predictive model** (for example, a logistic regression predicting `start` using the other variables). This could quantify the contribution of each factor while controlling for others. For instance, one could fit a logistic regression and find that low schooling and low growth remain significant predictors of war onset even when accounting for fractionalization and population.

However, building and interpreting such a model properly is complex and requires caution (plus knowledge of statistical modeling). We will introduce some of these techniques in the coming weeks.  For now, our focus is on exploratory analysis and understanding the relationships in the data.



**Your turn (Q9):** Suppose we find a strong correlation between a country's economic performance and civil war onset. What are some reasons we should be cautious about concluding that poor economic performance *causes* civil war?

**Your turn (Q10):** If you wanted to further test the relationship between multiple factors and civil war onset simultaneously (not one by one), what kind of analysis or model could you use? (Think in terms of statistical models you might know.)

*(Answer in Markdown: For Q9, discuss issues like potential reverse causality and confounders; for Q10, you might mention logistic regression or multivariate analysis.)*

**Answer (Q9):**



**Answer (Q10):**




## 8. Communicating Results



At this stage, we have gathered several insights from the data. Communicating these results effectively is as important as the analysis itself. We need to clearly summarize our findings, acknowledge limitations, and tell a coherent story about what the data shows.

In this section, we will outline how one might write up the results for an audience (e.g., in a report or presentation), and what caveats to mention. This includes discussing potential biases, data quality issues, and reminding that associations are not causation.

**Learning objectives for this section:** Learn how to compile findings into a clear narrative, discuss biases/confounders and data limitations, and practice data storytelling (turning numbers into insights with context).

Here are the key points we might communicate from our analysis:

- **Associated variables:** Some factors stood out as different between conflict periods and peaceful periods. For example, **education and economic growth** were notably lower in periods that experienced a civil war onset compared to peaceful periods. Countries/periods with new wars had on average ~30% male secondary schooling vs ~44% for others, and an average GDP growth around 0% (flat or slight recession) vs +1.7% in peaceful times. This suggests that economic underdevelopment and stagnation correlate with conflict outbreaks. On the social side, war onset periods were in countries with larger populations and higher ethnic/religious diversity on average (fractionalization index ~0.23 vs ~0.17 in peaceful cases).
- **Factors with little difference:** Interestingly, having one dominant ethnic group (the **dominance** index) did **not** show a clear relationship with war onset in our data – roughly half the cases in both war and peace groups had a dominant majority. So, simply having a majority group did not tilt the likelihood of conflict one way or the other here.
- **Biases and confounding factors:** We must be cautious interpreting these results:
    - *Data quality and missing data:* Many economic data points (like growth and schooling) were missing, especially for some war-torn countries or earlier years. This could bias our results if, say, the only countries with growth data during wars were a non-representative subset. Missing data might mean we underestimated or overestimated some differences.
    - *Timing and causality:* Our analysis used data from the same period as the war onset. This blurs cause and effect. For example, if a war started in 1975, the economy might have crashed because of the war – yet we recorded it as "low growth associated with war onset." In reality, the war didn’t start because growth was low; growth became low because war started. To disentangle this, ideally we would use data from *before* the war onset for the predictors, or look at longer trends leading up to war.
    - *Omitted variables:* We focused on a few variables, but many important factors are not in our dataset. Political regime type (democracy vs autocracy), external interventions, geographic terrain (mountains, etc.), colonial history, and so on could all influence conflict risk. If those correlate with our variables (for instance, maybe former colonies have both lower schooling and higher conflict risk), they could be underlying causes. Our analysis can’t separate those.
    - *Multiple comparisons:* We examined quite a few variables. By chance, some differences might appear just due to random variation. In a more rigorous analysis, we would adjust for this or use a model to evaluate multiple factors jointly, to ensure we’re not over-interpreting a fluke result.
- **Overall interpretation:** Within these caveats, the data suggests a story that aligns with some theories of civil conflict: countries that are less developed (lower education, economic struggles) and more socially fragmented tend to have a higher risk of civil war. However, these factors alone do not guarantee conflict – and conflict can erupt due to unique historical triggers or leadership decisions that are beyond what this data captures. For example, a nation might have low education and high diversity and remain peaceful if it has strong inclusive governance; conversely, a relatively well-off country could fall into conflict due to a political crisis. **No single factor causes civil war**; it's the combination of political, economic, and social pressures, often compounded by historical circumstances.
- **Communication style:** If presenting this to a general audience, we would avoid jargon like "t-test" or "p-value" and instead say "this difference is statistically significant, meaning it's likely a real difference and not just random chance in our sample." We would use clear visuals (perhaps a chart comparing education levels or growth rates for war vs peace cases) to illustrate the differences. And we’d stress that these are patterns in historical data, not destiny or simple cause-effect.

In summary, the data story might be: *"Civil wars between 1960 and 1995 were more likely to break out in larger, more diverse countries especially when those countries faced economic hardships and lower educational attainment. That said, these factors are pieces of a complex puzzle – every conflict has its unique context, and strong institutions or leadership can mitigate risks even when risk factors are present. Our analysis provides clues consistent with academic findings: development and social cohesion matter for peace, but correlation is not causation, and many other elements need to be considered when explaining why civil wars happen."*

*(The above is how we might wrap up our findings in a report. Now it’s your turn to articulate these points in your own words if you were writing the conclusion.)*



**Your turn (Q11):** Write a brief concluding summary of our analysis as if you were explaining to a reader or decision-maker. Highlight 2–3 key findings and mention at least one important caveat or limitation. Aim for a couple of well-structured paragraphs that tell the story of the data.

*(Use a Markdown cell for your response. Imagine this is the conclusion section of a report on this analysis.)*

**Answer (Q11):**





## 9. Capstone Mini-Project



Now that you’ve walked through the full data science pipeline on this civil war dataset, it's time for a mini-project! This is an opportunity to apply what you've learned to a new question or to dig deeper into the data.

Now that you’ve walked through the full data science pipeline on this civil war dataset, it's time for a mini-project! This is an opportunity to apply what you've learned to a new question or to dig deeper into the data. Below are a few options – please choose one or explore multiple if you’re curious:

**Option A: Explore a new factor (peace duration).** We touched on the `peace` column (months of peace since the last war) but didn't analyze it deeply. Investigate whether the length of peace seems to affect war onset. For example:

- Does a longer peace spell reduce the chance of another war, or is there evidence of conflicts recurring after a certain period?
- You could compare the average peace months in periods that stayed peaceful vs those that saw a new war. Or categorize peace months into "recent war" vs "long peace" and see how war onset frequencies differ.

**Option B: Cold War vs Post-Cold War analysis.** Divide the data into two subsets: 1960–1989 (the Cold War era) and 1990–1995 (post-Cold War era). Compare patterns between these periods. For instance:

- Were economic factors like growth and schooling as strongly associated with war onset during the Cold War as they were after? (You might find that the post-1990 conflicts had somewhat different profiles.)
- You can use filtering (e.g., `df[df['year'] < 1990]` vs `df[df['year'] >= 1990]`) and then replicate some of the group comparisons or counts for each era.

**Option C: Focus on a specific country or region.** Pick a country that experienced war(s) (for example, **Angola** or **Afghanistan** which had multiple onsets) and do a mini case study:

- Extract that country’s data (`df[df['country']=='Angola']`) and visualize its time series of various indicators. Do you see noticeable changes leading up to war onsets (e.g., dropping GDP growth, etc.)?
- You could plot Angola’s exports, growth, schooling over time and mark when wars started (perhaps using vertical lines or annotations).
- This gives practice with time series plotting and adds a narrative: e.g., *"Angola’s civil war resumed in 1975 amid a period of economic turmoil and low growth..."* etc.

For any option, make sure to:

- Clearly state the question you’re investigating.
- Write the code to perform the analysis (filtering, grouping, plotting, etc. as needed).
- Draw conclusions or interpretations from your findings.

The goal of these projects is to consolidate your EDA skills and encourage you to think like a data scientist: **ask questions, use the tools to answer them, and interpret the results**.

Good luck, and have fun exploring!
*(You can create new code and markdown cells below this point to work on the mini-project. There's no single "right" answer – it's all about the insights you derive.)*

## 10. Summary and Troubleshooting



Congratulations on making it through the tutorial! Let's recap what we did and learned, and also discuss some common issues (and their fixes) that you might encounter when doing similar data science work:

**Summary of Steps:**

1. **Data Loading:** We loaded the CSV file into a pandas DataFrame using `pd.read_csv`. We saw how to specify file paths and got our data into a usable structure (`df`).
2. **Data Inspection:** We used `df.head()`, `df.info()`, and `df.describe()` to understand the shape of the data, the types of variables, and to detect missing values or anomalies. This gave us an initial familiarity with the dataset.
3. **Data Cleaning:** We removed an unnecessary column (`Unnamed: 0`) and examined missing data. We discussed strategies for handling missing values (like leaving them, dropping, or imputing) and decided to proceed with caution, keeping missingness in mind.
4. **Exploratory Data Analysis (EDA):** We explored the distribution of our key outcome (`start` for war onset) across time and countries, using methods like `value_counts`, filtering, and `groupby`. We computed summary statistics (means) for groups (war vs peace) to see how different factors behaved.
5. **Feature Engineering:** We created new features (`population_millions` and `recession`) to make certain information more interpretable, showing how to add new columns based on existing data.
6. **Data Visualization:** We generated basic plots (bar charts, histograms, scatter plots, box plots) to visualize trends and distributions. Visualization helped confirm patterns (like the 1990s war spike, or the lower schooling in war cases) and provided an intuitive understanding of the data.
7. **Statistical Inference:** We performed simple statistical tests (t-tests) to check the significance of differences observed, and we reinforced the principle that correlation is not causation, discussing possible confounding factors and reverse causality.
8. **Communicating Results:** We practiced summarizing findings in plain language and noting limitations. We emphasized storytelling with data — turning the analysis into a narrative about what factors are associated with civil war onsets and what caution is needed.
9. **Capstone Project:** We proposed some further analysis ideas, encouraging independent exploration to solidify the skills learned.
10. **Troubleshooting:** (This section) We will now list a few common problems one might face and how to troubleshoot them.

**Common Issues & Troubleshooting Tips:**

- *Issue: File not found error when using `pd.read_csv`.*
    
    **Tip:** Check that the `ch.csv` file is in the correct directory or provide the full path to it.
    
    
- *Issue: Confusion with SettingWithCopyWarning when creating new columns.*
    
    **Tip:** This warning sometimes appears if you create a new column on a sliced DataFrame. Our approach `df['new_col'] = ...` on the original `df` is fine. If you ever encounter this warning, one fix is to use `.copy()` when sub-setting dataframes, or ensure you're assigning to the original DataFrame and not a view. In our tutorial, we didn't run into it explicitly, but it's a common pandas quirk.
    
- *Issue: Calculations seem off because of NaN values.*
    
    **Tip:** Remember that operations like `.mean()` skip NaNs by default, but if you are doing something like dividing counts, be mindful if some data was excluded. Use `df.dropna()` or specify `skipna` if needed. For instance, when calculating proportions, ensure the denominator is what you expect (did you want to include periods with ongoing wars or exclude them? we had to think about that in interpreting some percentages).
    
- *Issue: Plot labels overlap or chart is hard to read.*
    
    **Tip:** You can adjust figure size by adding `figsize=(width, height)` in the pandas plotting call. We did that for the histogram. For overlapping x-axis labels (e.g., country names), you might rotate them: `plt.xticks(rotation=45)` for example. Also trimming to top N items (like we did top 10 countries) helps readability.
    
- *Issue: Not sure what a function does or what parameters it takes.*
    
    **Tip:** Use the built-in `help(function)` or Google the pandas documentation. For example, `help(pd.DataFrame.groupby)` would show usage. Pandas has extensive docs and examples online.
    
- *Issue: Getting an error like "KeyError: 'XYZ'" when doing `df['XYZ']`.*
    
    **Tip:** This means the column 'XYZ' doesn't exist in `df` (maybe a typo in the column name, or you created it in a later cell but are trying to use it earlier). Double-check the spelling (`df.columns` list can help) and the order of cell execution. If you added a new column in one cell, you must run that cell before a subsequent cell that uses the new column.
    
- *Issue: Jupyter notebook state got messy (e.g., you edited code above and things stopped working).*
    
    **Tip:** It happens. If things behave oddly, try running the cells in order again (Kernel -> Restart & Run All, in Jupyter). Maintaining a clean sequential execution can resolve a lot of confusion, especially after making changes.
    
Finally, **remember that data analysis is an iterative process**. We often circle back to earlier steps (for example, after EDA you might realize you need another cleaning step, or after modeling you might do more feature engineering). Don't be afraid to revisit and refine.

We hope this in-class activity has given you a practical sense of performing data science tasks with pandas on a real-world dataset, and how to interpret and communicate the results. Happy data exploring!